# Risk-Aware Nordic Mushroom Species Recognition
## IKT460 Computer Vision — Class Project

**Research question:**
> *How well do ResNet-based architectures recognize Nordic wild mushroom species from field images, and can confidence-based abstention reduce dangerous confusions between edible and poisonous species?*

---

### Project design

| Model | Paper | Key innovation |
|-------|-------|----------------|
| **ResNet-50** | He et al., CVPR 2016 *(Best Paper)* | Residual connections; solves vanishing gradient in deep CNNs |
| **ResNeXt-50** | Xie et al., CVPR 2017 | Aggregated residual transformations (grouped convolutions) |
| **DenseNet-121** | Huang et al., CVPR 2017 | Dense connections; every layer feeds all later layers |
| **SE-ResNet-50** | Hu et al., CVPR 2018 *(Best Paper)* | Squeeze-and-Excitation; channel-wise attention mechanism |

**Why species-first, not binary edible/poisonous?**  
The real visual challenge in the field is *which species is this*, not just safe/unsafe. A species-first pipeline is more realistic, the accuracy numbers are more meaningful, and the risk labels remain correct even when taxonomy is updated. Safety is an analysis layer applied after prediction.

**Dataset:** Danish Fungi 2020 (DF20) — fine-grained benchmark, 295 species, built from the Atlas of Danish Fungi. Authors explicitly note it is challenging and suitable for comparing ImageNet-fine-tuned models.

**Society relevance:** Norway's Poisons Information Centre (Helsenorge) warns that poisonous mushrooms can be confused with edible species and advises eating only mushrooms you are **100% certain** are safe.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
from IPython.display import Image, display

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "df20_species_project"

# ── Dataset paths ─────────────────────────────────────────────────────────────
# Set one of the two options below before running training.
# Option A: single combined metadata file with a 'split' column
METADATA_PATH = None          # e.g. Path("/data/df20/DF20_metadata.csv")

# Option B: separate train / val / test CSV files
TRAIN_METADATA = None         # e.g. Path("/data/df20/DF20_train_metadata.csv")
VAL_METADATA   = None         # e.g. Path("/data/df20/DF20_val_metadata.csv")
TEST_METADATA  = None         # e.g. Path("/data/df20/DF20_test_metadata.csv")

IMAGES_ROOT    = None         # e.g. Path("/data/df20/images_300px")

# ── Training gate ─────────────────────────────────────────────────────────────
# Set to True ONLY when you want to launch training from inside this notebook.
RUN_TRAINING = False

print(f"Project root : {PROJECT_ROOT}")
print(f"Output dir  : {OUTPUT_DIR}")
print(f"Results exist: {(OUTPUT_DIR / 'results.csv').exists()}")

---
## 0. Training (optional)

Set `RUN_TRAINING = True` and configure the paths above to launch training.  
For a full run use `--top-species 100 --epochs 15`. A quick smoke test uses `--top-species 10 --epochs 3`.

Alternatively run from the terminal:
```bash
python scripts/train_df20_models.py \
    --train-metadata-path /data/df20/DF20_train_metadata.csv \
    --val-metadata-path   /data/df20/DF20_val_metadata.csv \
    --test-metadata-path  /data/df20/DF20_test_metadata.csv \
    --images-root         /data/df20/images_300px \
    --models resnet50 resnext50_32x4d densenet121 seresnet50 \
    --top-species 100 --min-images-per-species 30 \
    --epochs 15 --batch-size 32 \
    --output-dir outputs/df20_species_project
```

In [ ]:
if RUN_TRAINING:
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "train_df20_models.py"),
        "--models", "resnet50", "resnext50_32x4d", "densenet121", "seresnet50",
        "--top-species", "100",
        "--min-images-per-species", "30",
        "--epochs", "15",
        "--batch-size", "32",
        "--output-dir", str(OUTPUT_DIR),
    ]
    if METADATA_PATH:
        cmd += ["--metadata-path", str(METADATA_PATH)]
    else:
        if TRAIN_METADATA: cmd += ["--train-metadata-path", str(TRAIN_METADATA)]
        if VAL_METADATA:   cmd += ["--val-metadata-path",   str(VAL_METADATA)]
        if TEST_METADATA:  cmd += ["--test-metadata-path",  str(TEST_METADATA)]
    if IMAGES_ROOT:
        cmd += ["--images-root", str(IMAGES_ROOT)]

    print("Launching training...")
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("RUN_TRAINING = False — reading saved outputs only.")

---
## 1. Dataset & experiment overview

In [ ]:
meta_path = OUTPUT_DIR / "metadata.json"
results_path = OUTPUT_DIR / "results.csv"

if not meta_path.exists():
    raise FileNotFoundError(
        f"No metadata found at {meta_path}.\n"
        "Run training first (set RUN_TRAINING = True) or point OUTPUT_DIR to an existing run."
    )

with meta_path.open(encoding="utf-8") as f:
    meta = json.load(f)

cfg = meta["config"]
splits = meta["split_sizes"]
counts = meta.get("selected_species_counts_train", {})

print("=" * 55)
print("EXPERIMENT SUMMARY")
print("=" * 55)
print(f"  Split strategy : {meta['split_strategy']}")
print(f"  Species trained: {meta['num_classes']}")
print(f"  Train samples  : {splits['train']:,}")
print(f"  Val samples    : {splits['val']:,}")
print(f"  Test samples   : {splits['test']:,}")
print(f"  Total selected : {meta['selected_num_samples']:,}")
print(f"  Raw loaded     : {meta['raw_num_samples']:,}")
print(f"  Risk map size  : {meta['risk_map_size']} species")
print(f"  Device used    : {meta['device']}")
print()
print(f"  Image size     : {cfg['image_size']}px")
print(f"  Batch size     : {cfg['batch_size']}")
print(f"  Epochs max     : {cfg['epochs']}")
print(f"  Learning rate  : {cfg['learning_rate']}")
print(f"  Label smoothing: {cfg['label_smoothing']}")
print(f"  Weighted sampl.: {cfg['use_weighted_sampler']}")
print(f"  Pretrained     : {cfg['pretrained']}")

In [ ]:
if counts:
    count_series = pd.Series(counts, name="train_images")
    count_series.index.name = "species_key"

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Distribution of images per species
    axes[0].hist(count_series.values, bins=30, color="#386641", edgecolor="white")
    axes[0].set_xlabel("Training images per species")
    axes[0].set_ylabel("Number of species")
    axes[0].set_title("Long-tail distribution of training images")
    axes[0].axvline(count_series.mean(), color="#bc4749", linestyle="--",
                    label=f"Mean = {count_series.mean():.0f}")
    axes[0].axvline(count_series.median(), color="#f2e8cf", linestyle="--",
                    label=f"Median = {count_series.median():.0f}")
    axes[0].legend()

    # Top-20 species by count
    top20 = count_series.nlargest(20)
    axes[1].barh(top20.index[::-1], top20.values[::-1], color="#6a994e")
    axes[1].set_xlabel("Training images")
    axes[1].set_title("Top 20 species by training count")

    fig.tight_layout()
    plt.show()

    print(f"\nTotal species: {len(count_series)}")
    print(f"Image count — min: {count_series.min()}, median: {count_series.median():.0f}, "
          f"max: {count_series.max()}, mean: {count_series.mean():.1f}")

In [ ]:
# Load risk map and compute coverage over the trained species
import csv

risk_map_path = PROJECT_ROOT / cfg["risk_map_path"]
risk_map: dict[str, str] = {}
if risk_map_path.exists():
    with risk_map_path.open(encoding="utf-8") as f:
        for row in csv.DictReader(f):
            key = row.get("species_key", "").strip().lower()
            label = row.get("risk_label", "unknown").strip().lower()
            if key:
                risk_map[key] = label

class_keys = meta.get("class_keys", [])
risk_counts: dict[str, int] = {}
for k in class_keys:
    label = risk_map.get(k, "unknown")
    risk_counts[label] = risk_counts.get(label, 0) + 1

print("Risk label distribution over trained species:")
for label, cnt in sorted(risk_counts.items(), key=lambda x: x[1], reverse=True):
    pct = 100 * cnt / len(class_keys) if class_keys else 0
    bar = "█" * int(pct / 2)
    print(f"  {label:22s} {cnt:4d} ({pct:5.1f}%) {bar}")

covered = sum(v for k, v in risk_counts.items() if k != "unknown")
print(f"\nRisk coverage: {covered}/{len(class_keys)} species = {100*covered/len(class_keys):.1f}%")

---
## 2. Model comparison results

All four models are evaluated on the held-out test set. Key metrics:

- **Top-1 accuracy** — exact species correct
- **Top-3 accuracy** — true species in top 3 predictions
- **Macro F1** — unweighted average F1 across all species; penalises poor performance on rare classes
- **Balanced accuracy** — mean per-class recall; handles class imbalance
- **Dangerous error rate** — fraction of truly *unsafe* species predicted as *safe*

In [ ]:
if not results_path.exists():
    raise FileNotFoundError(f"No results found at {results_path}")

results = pd.read_csv(results_path)

display_cols = [
    "display_name", "parameters",
    "test_accuracy", "test_top3_accuracy",
    "test_balanced_accuracy", "test_f1_macro",
    "risk_coverage", "risk_accuracy", "dangerous_error_rate",
]
results_display = (
    results[display_cols]
    .sort_values("test_f1_macro", ascending=False)
    .reset_index(drop=True)
)
results_display["parameters"] = results_display["parameters"].apply(lambda x: f"{int(x):,}")

float_cols = [c for c in display_cols if c not in ("display_name", "parameters")]

pd.set_option("display.float_format", "{:.3f}".format)
display(results_display.style
    .highlight_max(subset=["test_accuracy", "test_top3_accuracy", "test_f1_macro",
                            "test_balanced_accuracy", "risk_accuracy"], color="#d4edda")
    .highlight_min(subset=["dangerous_error_rate"], color="#d4edda")
    .highlight_max(subset=["dangerous_error_rate"], color="#f8d7da")
    .format({c: "{:.3f}" for c in float_cols})
)

In [ ]:
comparison_plot = OUTPUT_DIR / "model_comparison.png"
if comparison_plot.exists():
    display(Image(filename=str(comparison_plot), width=900))
else:
    print(f"Plot not found at {comparison_plot}")

In [ ]:
best = results.sort_values("test_f1_macro", ascending=False).iloc[0]
worst = results.sort_values("test_f1_macro", ascending=False).iloc[-1]

print("Key findings from model comparison:")
print(f"  Best macro F1  : {best['display_name']} ({best['test_f1_macro']:.3f})")
print(f"  Worst macro F1 : {worst['display_name']} ({worst['test_f1_macro']:.3f})")
print(f"  F1 gap         : {best['test_f1_macro'] - worst['test_f1_macro']:.3f}")
print()

safest = results.sort_values("dangerous_error_rate").iloc[0]
riskiest = results.sort_values("dangerous_error_rate", ascending=False).iloc[0]
print(f"  Lowest  dangerous error rate: {safest['display_name']} ({safest['dangerous_error_rate']:.3f})")
print(f"  Highest dangerous error rate: {riskiest['display_name']} ({riskiest['dangerous_error_rate']:.3f})")
print()
print(f"  Risk map coverage: {best['risk_coverage']:.1%} of test samples have a known risk label")

---
## 3. Training history per model

Loss and accuracy curves show whether models converge and whether there is overfitting.

In [ ]:
model_names = list(results["model_name"])
history_plots = [(name, OUTPUT_DIR / f"{name}_history.png") for name in model_names]

available = [(name, p) for name, p in history_plots if p.exists()]
if not available:
    print("No training history plots found.")
else:
    fig, axes = plt.subplots(len(available), 1, figsize=(13, 4 * len(available)))
    if len(available) == 1:
        axes = [axes]
    for ax, (name, plot_path) in zip(axes, available):
        img = mpimg.imread(str(plot_path))
        ax.imshow(img)
        ax.axis("off")
    fig.tight_layout(pad=0.5)
    plt.show()

---
## 4. Confidence-based abstention

Instead of always forcing a prediction, a model can **abstain** on low-confidence samples and suggest the user consult an expert. This is the core safety mechanism.

- **Coverage** = fraction of test samples the model answers (higher = answers more)
- **Dangerous error rate** = fraction of truly unsafe species predicted safe (lower = safer)

An ideal curve shows dangerous error rate dropping faster than coverage as threshold rises.

In [ ]:
abstention_frames = []
for name in model_names:
    path = OUTPUT_DIR / f"{name}_abstention.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame.insert(0, "model", name)
        abstention_frames.append(frame)

if abstention_frames:
    abstention_df = pd.concat(abstention_frames, ignore_index=True)
    pd.set_option("display.float_format", "{:.3f}".format)
    display(abstention_df)
else:
    print("No abstention CSVs found.")

In [ ]:
abstention_plot = OUTPUT_DIR / "abstention_comparison.png"
if abstention_plot.exists():
    display(Image(filename=str(abstention_plot), width=900))
else:
    print(f"Abstention plot not found at {abstention_plot}")

In [ ]:
if abstention_frames:
    print("Abstention summary — effect of highest threshold tested:")
    print()
    for name in model_names:
        frame = abstention_df[abstention_df["model"] == name]
        if frame.empty:
            continue
        low_thresh = frame.sort_values("threshold").iloc[0]
        high_thresh = frame.sort_values("threshold").iloc[-1]
        coverage_drop = low_thresh["coverage"] - high_thresh["coverage"]
        danger_drop = low_thresh["retained_dangerous_error_rate"] - high_thresh["retained_dangerous_error_rate"]
        print(f"  {name}")
        print(f"    Coverage  : {low_thresh['coverage']:.2%} → {high_thresh['coverage']:.2%} (−{coverage_drop:.2%})")
        print(f"    Danger    : {low_thresh['retained_dangerous_error_rate']:.3f} → {high_thresh['retained_dangerous_error_rate']:.3f} (−{danger_drop:.3f})")
        print()

---
## 5. Safety analysis: risk confusion matrices

The key safety question is not just species accuracy but whether **dangerous confusions** occur — i.e. a truly unsafe species (poisonous or deadly) predicted as safe (edible or conditionally edible).

These matrices show model predictions at the **risk level**, not species level.

In [ ]:
risk_plots = [(name, OUTPUT_DIR / f"{name}_risk_confusion.png") for name in model_names]
available_risk = [(name, p) for name, p in risk_plots if p.exists()]

if not available_risk:
    print("No risk confusion plots found.")
else:
    cols = min(2, len(available_risk))
    rows = (len(available_risk) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 5.5 * rows))
    axes = np.array(axes).ravel()
    for ax, (name, plot_path) in zip(axes, available_risk):
        img = mpimg.imread(str(plot_path))
        ax.imshow(img)
        ax.axis("off")
    for ax in axes[len(available_risk):]:
        ax.axis("off")
    fig.tight_layout(pad=0.5)
    plt.show()

In [ ]:
# Load per-model prediction tables and compute dangerous confusion details
UNSAFE = {"poisonous", "deadly"}
SAFE   = {"edible", "conditionally_edible"}

print("Dangerous confusions (unsafe predicted as safe) per model:")
print("─" * 70)
for name in model_names:
    pred_path = OUTPUT_DIR / f"{name}_predictions.csv"
    if not pred_path.exists():
        print(f"  {name}: predictions file not found")
        continue

    preds = pd.read_csv(pred_path)
    # Only rows where we know the true risk
    known = preds[preds["true_risk"].isin(UNSAFE | SAFE)].copy()
    dangerous = known[
        known["true_risk"].isin(UNSAFE) & known["pred_risk"].isin(SAFE)
    ]

    print(f"\n  {name}")
    print(f"    Test samples with known risk : {len(known)}")
    print(f"    Truly unsafe samples         : {known['true_risk'].isin(UNSAFE).sum()}")
    print(f"    Dangerous errors             : {len(dangerous)}")
    if len(dangerous) > 0:
        top = (
            dangerous.groupby(["true_species_name", "pred_species_name", "true_risk"])
            .size()
            .reset_index(name="count")
            .sort_values("count", ascending=False)
            .head(5)
        )
        for _, row in top.iterrows():
            print(f"      [{row['true_risk']}] {row['true_species_name']} → {row['pred_species_name']} ({int(row['count'])}x)")

---
## 6. Per-species breakdown

Which species are hardest? Are the difficult species systematically the rarer ones (long-tail effect) or the visually similar ones?

In [ ]:
# Use the best model by F1
best_model_name = results.sort_values("test_f1_macro", ascending=False).iloc[0]["model_name"]
print(f"Analysing per-class metrics for best model: {best_model_name}")

per_class_path = OUTPUT_DIR / f"{best_model_name}_per_class_metrics.csv"
if not per_class_path.exists():
    print(f"File not found: {per_class_path}")
else:
    per_class = pd.read_csv(per_class_path)

    # Attach risk label
    per_class["risk"] = per_class["class_key"].map(
        lambda k: risk_map.get(k, "unknown")
    )

    print(f"\nBottom 15 species by F1 (hardest to recognise):")
    bottom = per_class.sort_values("f1").head(15)[["class_name", "f1", "recall", "support", "risk"]]
    display(bottom.style.format({"f1": "{:.3f}", "recall": "{:.3f}"})
            .background_gradient(subset=["f1"], cmap="RdYlGn"))

    print(f"\nTop 15 species by F1 (easiest to recognise):")
    top = per_class.sort_values("f1", ascending=False).head(15)[["class_name", "f1", "recall", "support", "risk"]]
    display(top.style.format({"f1": "{:.3f}", "recall": "{:.3f}"})
            .background_gradient(subset=["f1"], cmap="RdYlGn"))

In [ ]:
if per_class_path.exists():
    RISK_COLORS = {
        "edible": "#386641",
        "conditionally_edible": "#6a994e",
        "poisonous": "#e07a5f",
        "deadly": "#bc4749",
        "unknown": "#aaaaaa",
    }
    colors = per_class["risk"].map(lambda r: RISK_COLORS.get(r, "#aaaaaa"))

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.scatter(per_class["support"], per_class["f1"], c=colors, alpha=0.7, s=40)
    ax.set_xlabel("Test support (images)")
    ax.set_ylabel("F1 score")
    ax.set_title(f"{best_model_name} — F1 vs. test support, coloured by risk")

    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=9, label=r)
        for r, c in RISK_COLORS.items()
    ]
    ax.legend(handles=legend_elements, title="Risk label", loc="lower right")
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

In [ ]:
# Top confusion pairs — highlight dangerous ones
print(f"Top confusion pairs for {best_model_name}:\n")
confusion_path = OUTPUT_DIR / f"{best_model_name}_top_confusions.csv"
if confusion_path.exists():
    confusions = pd.read_csv(confusion_path)

    # Attach risk labels
    confusions["true_risk"] = confusions["true_species_name"].str.lower().str.replace(" ", "_").map(
        lambda k: risk_map.get(k, "unknown")
    )
    confusions["pred_risk"] = confusions["pred_species_name"].str.lower().str.replace(" ", "_").map(
        lambda k: risk_map.get(k, "unknown")
    )
    confusions["dangerous"] = (
        confusions["true_risk"].isin(UNSAFE) & confusions["pred_risk"].isin(SAFE)
    )

    def highlight_dangerous(row):
        return ["background-color: #f8d7da" if row["dangerous"] else "" for _ in row]

    display(
        confusions.head(20).style.apply(highlight_dangerous, axis=1)
        .format({"count": "{:d}"})
    )
    print("\nRed rows = dangerous confusions (unsafe predicted as safe)")
else:
    print(f"File not found: {confusion_path}")

---
## 7. Key findings and conclusions

In [ ]:
print("=" * 65)
print("FINAL RESULTS SUMMARY")
print("=" * 65)
print()

for _, row in results.sort_values("test_f1_macro", ascending=False).iterrows():
    print(f"  {row['display_name']:<22}")
    print(f"    Top-1 acc       : {row['test_accuracy']:.3f}")
    print(f"    Top-3 acc       : {row['test_top3_accuracy']:.3f}")
    print(f"    Macro F1        : {row['test_f1_macro']:.3f}")
    print(f"    Balanced acc    : {row['test_balanced_accuracy']:.3f}")
    print(f"    Dangerous errors: {row['dangerous_error_rate']:.3f} "
          f"({int(row['dangerous_errors'])} absolute)")
    print()

print("─" * 65)
print("Interpretation guide:")
print("  - Macro F1 is the primary metric on a long-tailed dataset")
print("  - Top-3 accuracy shows whether the correct species is")
print("    at least in the top candidates — relevant for expert use")
print("  - Dangerous error rate measures the real-world safety risk")
print("  - Abstention analysis shows the coverage/safety tradeoff")

### Discussion points for the report

1. **Do later architectures consistently outperform ResNet-50?**  
   Compare F1 across all four models. Are the improvements consistent across both common and rare species, or only on well-represented classes?

2. **Does top-1 and macro F1 tell the same story?**  
   On a long-tailed dataset, top-1 accuracy is dominated by frequent species. Macro F1 treats all species equally. A large gap between the two indicates that rare species are poorly handled.

3. **How many samples are covered by the risk map?**  
   DF20 has 295 species; the curated risk map covers common Nordic species. Coverage below 100% means some safety analysis is not possible — acknowledge this as a limitation.

4. **Does abstention meaningfully reduce dangerous errors?**  
   If the dangerous error rate drops significantly before coverage drops too much, abstention is a viable safety strategy. If coverage collapses before danger reduces, the model's confidence is poorly calibrated.

5. **Which species cause the most dangerous confusions?**  
   Species-level analysis reveals the hardest cases. Are dangerous confusions driven by visual similarity (e.g. chanterelle vs. false chanterelle) or by data scarcity?

6. **Why species-first matters:**  
   Training on coarse edible/poisonous folders conflates visually distinct species. A species-first model can be updated with new risk mappings without retraining.

### Safety disclaimer

> **This project is for research and coursework only. The models must not be used to decide whether a wild mushroom is safe to eat. Always consult an expert mycologist.**